# GeoRAG — תיקון המחברת הקיימת ל־v2

המחברת שלמה: כל התיקונים כבר משולבים בתאים. פתחו ב־Colab עם GPU והריצו לפי הסדר באמצעות Run all. אין צורך להוסיף תאים מתשובות קודמות. אין במחברת חיבור ל־Google Drive או בקשת הרשאה אליו; הקבצים נשמרים מקומית ב־/content/GeoRAG_NER וה־ZIP יורד למחשב בסיום.

ברירת המחדל היא PROCESS_ALL=True, עם RESELECT_V1_ZIP=None ו־RESUME_RUN_DIR=None. במקור שנבדק יש 36,335 פסקאות כשירות ב־428 מאמרים; הספירה מתבצעת מחדש בזמן הריצה. SAMPLE_SIZE=1000 נשאר להגדרת מצב בדיקה בלבד ואינו מגביל ריצה מלאה.

חוזה `ner-selected-2.0`: פלטי המודלים הם JSON במקטעים בלתי משתנים (checkpoints), עם תוצאות פסקה, חלונות, raw, validation ושגיאות. טבלת התאימות נשארת בת תשע עמודות; TIME/EVENT נמצאים בקבצי candidates עם status מפורש. ספי PER/DATE=0.50 הם כלל התחלתי שנבדק על Silver קיים, לא כיול סופי. אין שינוי בהחלטה שלא נבחר מודל סופי. מקור וחלוקת שורות v1/v4 נשמרים.

ברירת מחדל: כל הפסקאות הכשירות מכל DATA.json ב־commit המוגדר ב־GitHub. SEED/SAMPLE_SIZE מגבילים רק PROCESS_ALL=False. השמירה היא באחסון הזמני של Colab. חידוש אפשרי כל עוד קובצי הריצה עדיין קיימים; מחיקת ה־runtime עלולה למחוק אותם. כל shard סגור משמש checkpoint; מקטע שנקטע לפני commit מחושב שוב ואינו מופיע פעמיים. אין להריץ שני writers במקביל לאותה תיקייה. חידוש מאמת מקור, מדגם, הגדרות הסקה וגרסאות ספריות; שינוי מודל/גרסה דורש ריצה חדשה.

489 חריגות v1 כוללות end=len(window)+1 ונקודה סופית שאינה במקור. GLiNER מנרמל לעתים פיסוק בסוף טקסט; אין לקצר תחזית באופן שרירותי. התחזיות נשמרות כ־invalid ונחסמות מהתוצרים הנבחרים. v2 בודק גם גבולות חלון ולא רק פסקה.

שם התוצאה הראשית נשמר כ־{RUN_NAME}.csv, ובאותו בסיס גם Parquet ו־JSON, ללא תוספת _selected. RUN_NAME נבנה מ־FINAL_OUTPUT_PREFIX ומפרטי הריצה. רק תוצר המועמדים מקבל _candidates.

ב־v1 ה־JSON המשולב הכיל selected_mentions במערך יחיד. ב־v2 קובצי JSON הראשיים הם אינדקסים עם record_count ו־parts; הרשומות המפורטות נמצאות בקובצי JSON במקטעים. השינוי מאפשר יצוא ואימות בזיכרון חסום. צרכני JSON צריכים לעבור על parts; צרכני טבלת תשע העמודות יכולים להמשיך לקרוא CSV/Parquet. candidates CSV/Parquet מוסיפים selection_status, selected_model, semantic_validation ו־verified_event_fact. אין הצגה של התאמה סמנטית מאומתת גם בטבלת האזכורים.

לשחזור הבחירה מה־ZIP הישן, הגדירו RESELECT_V1_ZIP לנתיב ZIP. מצב זה טוען את מקור ומדגם הריצה הישנה ללא inference ושומר תוצר v2 חדש. הוא אינו הרצה מלאה. טעינת legacy JSON נעשית מודל אחד בכל פעם לצורך הסבה למקטעים; בריצה מלאה inference ויצוא עובדים במקטעים.


### 1 — התקנות

ללא שינוי מ־v1; הריצו תחילה ב־runtime חדש.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
import sys
import subprocess
import time
import json
import secrets
from contextlib import contextmanager

# Cell 1 starts a new execution; repeating other cells preserves their attempts.
CELL_TIMINGS = []
TIMING_EXECUTION_ID = secrets.token_hex(12)
ACTIVE_TIMING_RUN_NAME = None
INSTALLATION_LOG = ""

def print_cell_timings():
    print("\nCELL TIMINGS — measured code blocks (logging/download bookkeeping excluded)")
    for row in CELL_TIMINGS:
        print(f"{row['title']} | {row['seconds']:.2f}s | {row['minutes']:.2f}min | {row['status']}")
    print(f"Total measured: {sum(row['seconds'] for row in CELL_TIMINGS):.2f}s")

@contextmanager
def cell(name):
    started = time.perf_counter()
    at_started = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    status, error = "success", None
    try:
        yield
    except BaseException as exc:
        status, error = "failed", f"{type(exc).__name__}: {exc}"
        raise
    finally:
        elapsed = time.perf_counter() - started
        CELL_TIMINGS.append({"attempt": len(CELL_TIMINGS)+1, "title": name,
            "execution_id": TIMING_EXECUTION_ID, "run_name": ACTIVE_TIMING_RUN_NAME,
            "seconds": elapsed, "minutes": elapsed/60, "status": status,
            "error": error, "at_started_utc": at_started,
            "at_finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())})
        print(f"{'SUCCESS' if status == 'success' else 'FAILED'} {name}: "
              f"{error or 'completed'}; {elapsed:.2f}s ({elapsed/60:.2f}min)", flush=True)
        if status == "failed":
            print_cell_timings()

# Runs in a fresh Python process, so installed metadata is not cached by the kernel.
DEPENDENCY_CHECK_CODE = r"""
import json, sys
from importlib.metadata import distribution, PackageNotFoundError
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from packaging.utils import canonicalize_name

def check_dependencies(roots, get_distribution=distribution):
    queue = [Requirement(value) for value in roots]
    visited, checked, errors = set(), set(), []
    while queue:
        req = queue.pop()
        name = canonicalize_name(req.name)
        try:
            dist = get_distribution(req.name)
        except PackageNotFoundError:
            errors.append(f"Missing dependency: {req}")
            continue
        if req.specifier and dist.version not in req.specifier:
            errors.append(f"{req}: installed {dist.version}")
        key = (name, tuple(sorted(req.extras)))
        if key in visited:
            continue
        visited.add(key)
        checked.add(name)
        requires_python = dist.metadata.get("Requires-Python")
        if requires_python and '.'.join(map(str, sys.version_info[:3])) not in SpecifierSet(requires_python):
            errors.append(f"{name} requires Python {requires_python}; actual {sys.version.split()[0]}")
        for value in dist.requires or []:
            child = Requirement(value)
            if child.marker is None or any(child.marker.evaluate({"extra": extra}) for extra in ["", *req.extras]):
                queue.append(child)
    return {"checked_packages": sorted(checked), "errors": sorted(set(errors))}

if __name__ == "__main__":
    report = check_dependencies(json.loads(sys.argv[1]))
    print(json.dumps(report))
    sys.exit(1 if report["errors"] else 0)
"""

with cell("1 — התקנות"):
    packages = ["transformers==4.49.0", "gliner2[local]==2.1.0",
        "huggingface-hub>=0.30,<1", "peft==0.14.0", "packaging>=24,<27",
        "pandas>=2.2,<3", "pyarrow>=18,<24", "jedi==0.19.2"]
    def installation_command(args):
        global INSTALLATION_LOG
        result = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        INSTALLATION_LOG += f"\nCOMMAND: {args}\n{result.stdout}\n"
        if result.stdout:
            print(result.stdout, flush=True)
        if result.returncode:
            raise RuntimeError(f"Installation command failed ({result.returncode}): {args}\n{result.stdout}")
    print("Python:", sys.version)
    installation_command([sys.executable, "-m", "pip", "install", "--quiet", *packages])
    full_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    INSTALLATION_LOG += "\nGLOBAL PIP CHECK:\n" + full_check.stdout
    print("GLOBAL PIP CHECK:\n" + full_check.stdout, flush=True)
    if full_check.returncode not in [0, 1]:
        raise RuntimeError(f"pip check itself failed: {full_check.stdout}")
    relevant = subprocess.run([sys.executable, "-c", DEPENDENCY_CHECK_CODE,
        json.dumps([*packages, "torch", "google-colab"])], text=True,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    INSTALLATION_LOG += "\nNOTEBOOK DEPENDENCY CHECK:\n" + relevant.stdout + relevant.stderr
    if relevant.returncode:
        raise RuntimeError("Notebook dependency conflict. Full details:\n" + relevant.stdout + relevant.stderr)
    DEPENDENCY_REPORT = {"global_pip_check_returncode": full_check.returncode,
        "global_pip_check_output": full_check.stdout, "notebook_dependencies": json.loads(relevant.stdout)}
    if full_check.returncode:
        print("WARNING: global conflicts were reported, but the notebook dependency closure passed. "
              "These diagnostics will be retained; model imports are checked in cell 2.")


### 2 — ייבואים

נוספו ספריות תקן לכתיבה במקטעים ולשמירה אטומית; אין framework חדש.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("2 — imports and environment"):
    import os, csv, copy, inspect, shutil
    import gc, hashlib, io, logging, math, platform, random, re, tempfile, zipfile
    from collections import Counter, defaultdict
    from datetime import datetime, timezone
    from importlib.metadata import version
    from pathlib import Path
    from zoneinfo import ZoneInfo
    import pandas as pd
    import pyarrow as pa
    import pyarrow.parquet as pq
    import torch
    from gliner2 import AutoExtractor
    from google.colab import files
    from huggingface_hub import model_info, snapshot_download
    from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
    VERSIONS = {n: version(n) for n in ["torch", "transformers", "gliner2", "peft",
        "huggingface-hub", "pandas", "pyarrow", "numpy", "tokenizers"]}
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    log = logging.getLogger("ner_selected_v1")
    print(json.dumps(VERSIONS, indent=2))
    print("Python:", platform.python_version(), "CUDA:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    if torch.cuda.is_available():
        print("GPU free/total bytes:", torch.cuda.mem_get_info())


### 3 — הגדרות v2

ברירת מחדל מלאה, ספי בחירה נפרדים, שמירה מקומית ב־Colab ופרמטר חידוש.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("3 — configuration"):
    REPO_URL = "https://github.com/danielbr9382/Corpus2GeoRAG_Pacific.git"
    SOURCE_RELATIVE_PATH = "Data/DATA.json"
    SOURCE_COMMIT = "abb6fd24a6446ad52e488f27a401f17a471c2efc"
    MODELS = {
        "bert": {"id": "dslim/bert-base-NER", "revision": "d1a3e8f13f8c3566299d95fcfc9a8d2382a9affc"},
        "gliner": {"id": "fastino/gliner2.5-base-v1", "revision": "54785d51df8d86a0d8f2eb212fec15871aad2c4d"},
    }
    SEED = 17
    SAMPLE_SIZE = 1000  # paragraphs, not articles
    PROCESS_ALL = True
    RUN_RANDOM_ID = None  # None generates a new ID; does not affect sampling
    OUTPUT_ROOT = Path("/content/GeoRAG_NER")
    CHECKPOINT_PARAGRAPHS = 25
    RESUME_RUN_DIR = None  # existing local run directory, or None for a new run
    RESELECT_V1_ZIP = None  # optional: existing v1 ZIP, selection only; no inference
    FINAL_OUTPUT_PREFIX = "georag_selected_v2"
    ADD_BERT_PER = True
    BERT_PER_MIN_SCORE = 0.90
    EVENT_HIGH_PRIORITY_SCORE = 0.90
    GLINER_THRESHOLD = 0.01  # decoding floor, independent of selection thresholds
    GLINER_LABELS = ["person", "organization", "military unit", "location",
        "geopolitical entity", "facility", "nationality or political group", "event",
        "date", "time", "weapon", "vehicle", "product", "work of art", "law",
        "language", "money", "quantity", "percent", "ordinal", "cardinal",
        "miscellaneous named entity"]
    LABEL_MAP = {
        "PER": "PER", "LOC": "LOC", "ORG": "ORG", "MISC": "MISC",
        "person": "PER", "organization": "ORG", "military unit": "ORG",
        "location": "LOC", "geopolitical entity": "LOC", "facility": "LOC",
        "event": "EVENT", "date": "DATE", "time": "TIME",
        **{k: "MISC" for k in ["nationality or political group", "weapon", "vehicle",
            "product", "work of art", "law", "language", "money", "quantity", "percent",
            "ordinal", "cardinal", "miscellaneous named entity"]},
    }
    SELECTION_THRESHOLDS = {"PER": 0.50, "DATE": 0.50, "LOC": 0.0, "ORG": 0.0, "MISC": 0.0}
    # Initial v2 rule, not final calibration for the full corpus.
    PRIMARY = {"PER": "gliner", "LOC": "bert", "ORG": "bert", "DATE": "gliner",
               "TIME": "gliner", "EVENT": "gliner", "MISC": "bert"}
    WINDOWS = {n: {"max_encoded_tokens": 480, "max_words": 384, "overlap_words": 64}
               for n in MODELS}
    DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
    SCHEMA_VERSION = "ner-selected-2.0"
    SPLIT_VERSION = "nonempty_nonheading_lines_v1"  # unchanged from v4
    assert isinstance(SEED, int) and isinstance(SAMPLE_SIZE, int) and SAMPLE_SIZE > 0
    assert re.fullmatch(r"[A-Za-z0-9_-]+", FINAL_OUTPUT_PREFIX), "Use letters, digits, _ or -"
    assert all(0 <= t <= 1 for t in [BERT_PER_MIN_SCORE, EVENT_HIGH_PRIORITY_SCORE, GLINER_THRESHOLD])
    assert all(0 <= w["overlap_words"] < w["max_words"] for w in WINDOWS.values())
    assert all(re.fullmatch(r"[0-9a-f]{40}", m["revision"]) for m in MODELS.values())
    if RUN_RANDOM_ID is None:
        RUN_RANDOM_ID = secrets.randbelow(900_000_000)+100_000_000
    assert isinstance(RUN_RANDOM_ID, int) and RUN_RANDOM_ID >= 0
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    assert isinstance(CHECKPOINT_PARAGRAPHS, int) and CHECKPOINT_PARAGRAPHS > 0
    assert all(0 <= x <= 1 for x in SELECTION_THRESHOLDS.values())
    CONFIG = {"schema_version": SCHEMA_VERSION, "split_version": SPLIT_VERSION,
        "selection_thresholds": SELECTION_THRESHOLDS, "checkpoint_paragraphs": CHECKPOINT_PARAGRAPHS,
        "repo_url": REPO_URL, "source_path": SOURCE_RELATIVE_PATH, "source_commit": SOURCE_COMMIT,
        "models": MODELS, "seed": SEED, "sample_size_requested": SAMPLE_SIZE,
        "process_all": PROCESS_ALL, "run_random_id": RUN_RANDOM_ID,
        "output_root": str(OUTPUT_ROOT), "final_output_prefix": FINAL_OUTPUT_PREFIX,
        "add_bert_per": ADD_BERT_PER, "bert_per_min_score": BERT_PER_MIN_SCORE,
        "event_high_priority_score": EVENT_HIGH_PRIORITY_SCORE, "gliner_threshold": GLINER_THRESHOLD,
        "gliner_labels": GLINER_LABELS, "label_map": LABEL_MAP, "primary_v2": PRIMARY,
        "windows": WINDOWS, "device": DEVICE, "aggregation_strategy": "average",
        "window_dedup": "same model/label/exact span; first window wins; preserve all raw",
        "selection_dedup": "same paragraph/group/exact span; no score averaging",
        "semantic_validation": "not performed; EVENT/TIME are candidates only"}
    MODEL_OUTPUTS = {}
    print("Run ID:", RUN_RANDOM_ID, "device:", DEVICE, "policy:", PRIMARY)


### 4 — כלי היסטים

כלי v1 נשמרו; נוספה בדיקת גבולות חלון ואבחון נקודה סינתטית.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell('4 — source, offsets and adapters'):
    def utc_now():
        return datetime.now(timezone.utc).isoformat()

    def digest(payload):
        return hashlib.sha256(payload).hexdigest()

    def digest_file(path):
        hasher = hashlib.sha256()
        with Path(path).open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                hasher.update(block)
        return hasher.hexdigest()

    def jsonable(value):
        if hasattr(value, "item"):
            return value.item()
        raise TypeError(type(value).__name__)

    def encoded(obj):
        return json.dumps(obj, ensure_ascii=False, sort_keys=True, allow_nan=False,
                          default=jsonable).encode("utf-8")

    def save_json(path, obj):
        with Path(path).open("xb") as handle:
            handle.write(encoded(obj))

    def load_json(path):
        return json.loads(Path(path).read_bytes())

    def progress(name, done, total, errors, started, last):
        bucket = done * 10 // total
        for mark in range(last+1, bucket+1):
            print(f"{name}: {mark*10}% reached; processed={done}/{total}; "
                  f"success={done-errors}; failed={errors}; elapsed={time.perf_counter()-started:.2f}s", flush=True)
        return bucket

    def text_units(title, article):
        offset, index = 0, 0
        for line in article.splitlines(keepends=True):
            text = line.rstrip("\r\n")  # שומרים רווחים ותווים מקוריים
            if text.strip() and not re.fullmatch(r"=+\s*.*?\s*=+", text.strip()):
                end = offset + len(text)
                assert article[offset:end] == text
                yield {"Source": title, "Index": index, "Text": text,
                       "start_document": offset, "end_document": end}
                index += 1
            offset += len(line)

    def windows(text, settings, measure, limit):
        words = list(re.finditer(r"\S+", text))
        i = 0
        budget = min(settings["max_encoded_tokens"], limit)
        while i < len(words):
            left = 0 if i == 0 else words[i].start()
            high = min(len(words), i + settings["max_words"])
            def right(j):
                return words[j].start() if j < len(words) else len(text)
            # הפחתה עד שהקידוד המלא נכנס; לא מפעילים truncation
            j = high
            while j > i and measure(text[left:right(j)]) > budget:
                j -= max(1, (j-i)//8)
            if j == i:
                raise ValueError(f"Single word at {left} exceeds encoded token budget {budget}")
            end = right(j)
            count = measure(text[left:end])
            assert count <= budget
            yield {"start": left, "end": end, "Text": text[left:end], "encoded_tokens": count}
            if j == len(words):
                break
            i = max(i + 1, j - settings["overlap_words"])

    def group(label):
        return LABEL_MAP.get(re.sub(r"^[BI]-", "", str(label)))

    def mention(p, name, prediction, base, ref):
        result = {"model": name, "model_id": MODELS[name]["id"],
            "Source": p["Source"], "Index": p["Index"], "sample_position": p["sample_position"],
            "Text": p["Text"], "source_sha256": SOURCE_META["sha256"],
            "source_commit": SOURCE_META["commit_sha"], "source_path": SOURCE_RELATIVE_PATH,
            "text_sha256": digest(p["Text"].encode()), "label_original": prediction.get("label"),
            "subtype": prediction.get("label"), "entity_group": group(prediction.get("label")),
            "score": None, "start": None, "end": None, "word": prediction.get("text"),
            "start_document": None, "end_document": None, "status_validation": "invalid",
            "error": None, "ref_output_raw": [ref], "prediction": prediction}
        try:
            score = prediction.get("score")
            if score is not None:
                score = float(score)
                if not math.isfinite(score) or not 0 <= score <= 1:
                    raise ValueError("Non-finite score")
            result["score"] = score
            if prediction.get("alignment_error"):
                raise ValueError(prediction["alignment_error"])
            a, b = prediction["start"], prediction["end"]
            if not isinstance(a, int) or not isinstance(b, int) or isinstance(a, bool) or isinstance(b, bool):
                raise ValueError("Offsets must be integers")
            a, b = base + a, base + b
            result.update(start=a, end=b)
            if not 0 <= a < b <= len(p["Text"]):
                raise ValueError("Span outside paragraph")
            surface = p["Text"][a:b]
            # HF subword rendering is diagnostic only; other adapters verify model surface
            if prediction.get("surface_checked", True) and surface != prediction.get("text"):
                raise ValueError("Model surface differs from original source slice")
            da, db = p["start_document"] + a, p["start_document"] + b
            if DATA[p["Source"]][da:db] != surface:
                raise ValueError("Document slice differs from paragraph slice")
            if not isinstance(prediction.get("label"), str) or not prediction["label"]:
                raise ValueError("Missing label")
            result.update(word=surface, score=score, start_document=da, end_document=db,
                          status_validation="valid" if result["entity_group"] else "pending")
            if result["entity_group"] is None:
                result["error"] = "Unmapped label; update LABEL_MAP explicitly"
            result["semantic_validation"] = "not_assessed"
        except Exception as exc:
            result["error"] = f"{type(exc).__name__}: {exc}"
        return result

    def deduplicate(items):
        output, seen = [], {}
        for item in items:
            key = (item["Source"], item["Index"], item["model"], item["start"],
                   item["end"], item["label_original"])
            if item["status_validation"] == "valid" and key in seen:
                seen[key]["ref_output_raw"].extend(item["ref_output_raw"])
            else:
                output.append(item)
                if item["status_validation"] == "valid":
                    seen[key] = item  # בחירה ראשונה לפי סדר חלונות; לא לפי ממוצע ציון
        return output

    def process_paragraph(p, name, predict, measure, limit):
        started, raw, found, errors = time.perf_counter(), [], [], []
        try:
            for wi, w in enumerate(windows(p["Text"], WINDOWS[name], measure, limit)):
                record = {"window_id": wi, **w, "at_processed": utc_now(), "output": None,
                          "error": None, "status_processing": "pending", "status_validation": "pending"}
                raw.append(record)
                try:
                    output = predict(w["Text"])
                    record["output"] = output  # שומרים לפני פירוש/ולידציה
                    output_errors = output.get("errors", []) if isinstance(output, dict) else []
                    errors.extend(output_errors)
                    ref = f"{ACTIVE_RAW_FILE}#/results/{ACTIVE_RAW_INDEX}/raw/{wi}/output"
                    predictions = adapt_output(name, output, w["Text"])
                    for ordinal, prediction in enumerate(predictions):
                        prediction = dict(prediction)
                        a, b = prediction.get("start"), prediction.get("end")
                        if isinstance(a, int) and isinstance(b, int) and not 0 <= a < b <= len(w["Text"]):
                            synthetic = (name == "gliner" and b == len(w["Text"])+1
                                and prediction.get("text", "").endswith("."))
                            prediction["alignment_error"] = (
                                "GLiNER synthetic trailing punctuation outside source window"
                                if synthetic else "Span outside source window")
                        item = mention(p, name, prediction, w["start"], ref)
                        item["raw_ordinal"] = ordinal
                        found.append(item)
                    record["status_processing"] = "error" if output_errors else "success"
                    record["error"] = "; ".join(output_errors) or None
                    record["status_validation"] = "invalid" if output_errors or any(
                        m["status_validation"] != "valid" for m in found if ref in m["ref_output_raw"]
                    ) else "valid"
                except Exception as exc:
                    record["error"] = f"{type(exc).__name__}: {exc}"
                    record["status_processing"] = "error"
                    record["status_validation"] = "invalid"
                    errors.append(record["error"])
                    log.error(json.dumps({"model": name, "sample_position": p["sample_position"],
                                          "window": wi, "error": record["error"]}))
        except Exception as exc:
            errors.append(f"{type(exc).__name__}: {exc}")
            log.error(json.dumps({"model": name, "sample_position": p["sample_position"], "error": errors[-1]}))
        before_dedup = len(found)
        found = deduplicate(found)
        invalid = [m for m in found if m["status_validation"] != "valid"]
        status = "error" if errors else "partial_invalid" if invalid else "success"
        return {"paragraph": p, "raw": raw, "mentions": found, "duplicates_resolved": before_dedup-len(found),
            "status_processing": status, "errors": errors + [m["error"] for m in invalid],
            "at_processed": utc_now(), "processing_seconds": time.perf_counter()-started}

    def release():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def tokenizer_measure(tokenizer):
        return lambda text: len(tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])


    def hf_revision(name):
        return model_info(MODELS[name]["id"], revision=MODELS[name]["revision"]).sha

    def adapt_output(name, output, text):
        if name == "bert":
            return [{"label": x["entity_group"], "start": int(x["start"]), "end": int(x["end"]),
                     "score": x.get("score"), "text": x.get("word"), "surface_checked": False}
                    for x in output]
        if name == "gliner":
            return [{"label": label, "start": x["start"], "end": x["end"],
                     "text": x["text"], "score": x.get("confidence")}
                    for label, values in output["entities"].items() for x in values]
        raise ValueError(f"No adapter registered: {name}")

    print("v4 source/window/offset helpers ready; raw API outputs retained, not full logits.")


### 5 — מקור וכיסוי

טוען GitHub, סופר את כל הפסקאות, יוצר ריצה או מאמת חידוש.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("5 — source, full coverage and resume"):
    LEGACY = None
    if RESELECT_V1_ZIP:
        legacy_path = Path(RESELECT_V1_ZIP)
        with zipfile.ZipFile(legacy_path) as z:
            assert z.testzip() is None
            def member(suffix):
                matches = [n for n in z.namelist() if n.endswith(suffix)]
                if len(matches) != 1:
                    raise ValueError(f"Ambiguous legacy member: {suffix}")
                return matches[0]
            payload = z.read(member("_source.json"))
            legacy_sample = json.loads(z.read(member("_sample.json")))
            LEGACY = {"path": str(legacy_path), "sha256": digest_file(legacy_path),
                "members": {n: member("_ner_"+n+".json") for n in MODELS}}
        SOURCE_META = legacy_sample["source"]
    elif RESUME_RUN_DIR:
        resume_root = Path(RESUME_RUN_DIR)
        frozen = load_json(resume_root/"run.json")
        payload = (resume_root/"source.json").read_bytes()
        SOURCE_META = frozen["source"]
        assert (SOURCE_META["repo_url"], SOURCE_META["path"], SOURCE_META["commit_sha"]) == (
            REPO_URL, SOURCE_RELATIVE_PATH, SOURCE_COMMIT), "Source settings differ from resumed run"
    else:
        checkout = Path(tempfile.mkdtemp(prefix="ner_source_", dir="/content"))/"repo"
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(checkout)], check=True)
        if not re.fullmatch(r"[0-9a-fA-F]{40}", SOURCE_COMMIT):
            raise ValueError("Pin SOURCE_COMMIT to a full Git SHA")
        subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", SOURCE_COMMIT], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
        commit = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
        tracked = subprocess.check_output(["git", "-C", str(checkout), "ls-tree", "-r", "--name-only", "HEAD"], text=True).splitlines()
        if SOURCE_RELATIVE_PATH not in tracked:
            raise FileNotFoundError(SOURCE_RELATIVE_PATH)
        payload = (checkout/SOURCE_RELATIVE_PATH).read_bytes()
        SOURCE_META = {"repo_url": REPO_URL, "path": SOURCE_RELATIVE_PATH,
            "commit_sha": commit, "sha256": digest(payload)}
    DATA = json.loads(payload)
    assert isinstance(DATA, dict) and DATA and all(isinstance(k, str) and isinstance(v, str) for k,v in DATA.items())
    assert digest(payload) == SOURCE_META["sha256"]
    population = [p for title in sorted(DATA) for p in text_units(title, DATA[title])]
    population.sort(key=lambda p: (p["Source"], p["start_document"], p["Index"]))
    TOTAL_ELIGIBLE = len(population)
    if LEGACY:
        SAMPLE = legacy_sample["paragraphs"]
        mode = "reselected"
    else:
        mode = "full" if PROCESS_ALL else "sample"
        if not PROCESS_ALL and not 0 < SAMPLE_SIZE <= TOTAL_ELIGIBLE:
            raise ValueError(f"Requested {SAMPLE_SIZE}; available {TOTAL_ELIGIBLE}")
        selected_population = population if PROCESS_ALL else random.Random(SEED).sample(population, SAMPLE_SIZE)
        SAMPLE = [{**p, "sample_position": i} for i,p in enumerate(selected_population)]
    for i,p in enumerate(SAMPLE):
        assert p["sample_position"] == i
        assert DATA[p["Source"]][p["start_document"]:p["end_document"]] == p["Text"]
    if mode == "full":
        assert len(SAMPLE) == TOTAL_ELIGIBLE
    SAMPLE_PAYLOAD = {"schema_version": SCHEMA_VERSION, "split_version": SPLIT_VERSION,
        "source": SOURCE_META, "mode": mode, "paragraphs": SAMPLE}
    SAMPLE_HASH = digest(encoded(SAMPLE_PAYLOAD))
    # SEED and SAMPLE_SIZE do not enter the full-mode inference identity.
    INFERENCE_CONFIG = {"schema_version": SCHEMA_VERSION, "source": SOURCE_META,
        "sample_sha256": SAMPLE_HASH, "models": MODELS, "windows": WINDOWS,
        "labels": GLINER_LABELS, "label_map": LABEL_MAP, "threshold_raw": GLINER_THRESHOLD,
        "aggregation": "average", "versions": VERSIONS, "device": DEVICE,
        "checkpoint_paragraphs": CHECKPOINT_PARAGRAPHS,
        "legacy_sha256": LEGACY["sha256"] if LEGACY else None}
    INFERENCE_HASH = digest(encoded(INFERENCE_CONFIG))
    CONFIG.update(mode=mode, actual_sample_size=len(SAMPLE), total_eligible=TOTAL_ELIGIBLE)
    CONFIG_HASH = digest(encoded(CONFIG))
    if RESUME_RUN_DIR:
        RUN_DIR = Path(RESUME_RUN_DIR)
        frozen = load_json(RUN_DIR/"run.json")
        assert frozen["inference_sha256"] == INFERENCE_HASH, "Resume mismatch: source/model/revision/config/library versions"
        assert load_json(RUN_DIR/"sample.json") == SAMPLE_PAYLOAD
        RUN_NAME = frozen["run_name"]
    else:
        date = datetime.now(ZoneInfo("Asia/Jerusalem")).strftime("%Y%m%d")
        size_part = f"full_n{len(SAMPLE)}" if mode == "full" else f"{mode}_seed{SEED}_n{len(SAMPLE)}"
        RUN_NAME = f"{FINAL_OUTPUT_PREFIX}_{size_part}_r{RUN_RANDOM_ID}_{date}"
        RUN_DIR = OUTPUT_ROOT/RUN_NAME
        RUN_DIR.mkdir(parents=True, exist_ok=False)
        with (RUN_DIR/"source.json").open("xb") as f:
            f.write(payload)
        save_json(RUN_DIR/"sample.json", SAMPLE_PAYLOAD)
        save_json(RUN_DIR/"run.json", {"schema_version": SCHEMA_VERSION, "run_name": RUN_NAME,
            "source": SOURCE_META, "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
            "inference_configuration": INFERENCE_CONFIG, "legacy": LEGACY})
    SESSION_ID = secrets.token_hex(8)
    for row in CELL_TIMINGS:
        row["run_name"] = RUN_NAME
    ACTIVE_TIMING_RUN_NAME = RUN_NAME
    PATHS = {"selftests": RUN_DIR/f"selftests_{SESSION_ID}.json"}
    EXPORT_DIR = RUN_DIR/"exports"/SESSION_ID
    EXPORT_DIR.mkdir(parents=True, exist_ok=False)
    ZIP_PATH = OUTPUT_ROOT/f"{RUN_NAME}_export_{SESSION_ID}.zip"
    with (EXPORT_DIR/"environment.txt").open("x", encoding="utf-8") as f:
        f.write(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
    with (EXPORT_DIR/"installation_log.txt").open("x", encoding="utf-8") as f:
        f.write(INSTALLATION_LOG)
    print("Mode:", mode, "articles:", len(DATA), "eligible paragraphs:", TOTAL_ELIGIBLE,
        "planned this run:", len(SAMPLE), "run directory:", RUN_DIR)
    print("Resume later using RESUME_RUN_DIR =", repr(str(RUN_DIR)))


### 5א — checkpoints וחידוש

סריקת מקטעים, הסבת ZIP ישן ואימות חתימות; כל מודל משלים רק את המקטעים החסרים.


In [ ]:
with cell("5a — checkpoint engine"):
    from contextlib import ExitStack
    csv.field_size_limit(2**31-1)

    def stream_digest(handle):
        h=hashlib.sha256()
        for block in iter(lambda:handle.read(1024*1024),b""):
            h.update(block)
        return h.hexdigest()

    @contextmanager
    def contextmanager_export_writers():
        with ExitStack() as stack:
            writers={}
            for kind,cols,schema in [("selected",COLUMNS,ARROW_SCHEMA),("candidates",CANDIDATE_COLUMNS,CANDIDATE_SCHEMA)]:
                cf=stack.enter_context(OUTPUT_FILES[kind+"_csv"].open("x",encoding="utf-8-sig",newline=""))
                pf=stack.enter_context(OUTPUT_FILES[kind+"_parquet"].open("xb"))
                cw=csv.DictWriter(cf,fieldnames=cols)
                cw.writeheader()
                pw=stack.enter_context(pq.ParquetWriter(pf,schema))
                writers[kind]=(cols,schema,cw,pw)
            yield writers

    def streaming_roundtrip(kind):
        cols=COLUMNS if kind=="selected" else CANDIDATE_COLUMNS
        def expected_rows():
            for part in load_json(OUTPUT_FILES[kind+"_json"])["parts"]:
                yield from load_json(EXPORT_DIR/part["path"])["records"]
        expected=iter(expected_rows())
        count=0
        with OUTPUT_FILES[kind+"_csv"].open(encoding="utf-8-sig",newline="") as cf:
            reader=csv.DictReader(cf)
            assert reader.fieldnames==cols
            pf=pq.ParquetFile(OUTPUT_FILES[kind+"_parquet"])
            assert pf.schema_arrow==(ARROW_SCHEMA if kind=="selected" else CANDIDATE_SCHEMA)
            for batch in pf.iter_batches(batch_size=1024):
                for row in batch.to_pylist():
                    c=next(reader);detail=next(expected)
                    for k in cols:
                        assert row[k]==detail.get(k)
                        want=row[k]
                        if want is None:
                            assert c[k]==""
                        elif k in {"start","end","Index"}:
                            assert int(c[k])==want
                        elif k=="score":
                            assert float(c[k])==want
                        else:
                            assert c[k]==str(want)
                    count+=1
            assert next(reader,None) is None and next(expected,None) is None
        return count
    def atomic_json(path, obj):
        """Single writer only. A .tmp file is never a committed checkpoint."""
        path = Path(path)
        if path.exists():
            raise FileExistsError(path)
        temp = path.with_name(path.name+"."+secrets.token_hex(6)+".tmp")
        with temp.open("xb") as f:
            f.write(encoded(obj)); f.flush(); os.fsync(f.fileno())
        if path.exists():
            raise FileExistsError(path)
        os.rename(temp, path)

    def shard_path(name, start):
        return RUN_DIR/"models"/name/f"part_{start:08d}.json"

    def load_shard(name, start):
        path = shard_path(name, start)
        data = load_json(path)
        assert data["inference_sha256"] == INFERENCE_HASH
        assert data["sample_sha256"] == SAMPLE_HASH and data["model"] == name
        expected = SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]
        assert [r["paragraph"] for r in data["results"]] == expected
        assert data["payload_sha256"] == digest(encoded(data["results"]))
        return data

    def scan_model(name):
        covered, totals = set(), Counter()
        for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
            if not shard_path(name, start).exists():
                continue
            data = load_shard(name, start)
            for r in data["results"]:
                pos = r["paragraph"]["sample_position"]
                assert pos not in covered
                covered.add(pos)
                totals[r["status_processing"]] += 1
        return covered, totals

    def model_complete(name):
        return len(scan_model(name)[0]) == len(SAMPLE)

    def run_model(name, predict, measure, limit, metadata):
        global ACTIVE_RAW_FILE, ACTIVE_RAW_INDEX
        folder = RUN_DIR/"models"/name
        folder.mkdir(parents=True, exist_ok=True)
        metadata = {**metadata, "device": DEVICE, "window": WINDOWS[name],
            "effective_encoded_limit": limit, "threshold": GLINER_THRESHOLD if name == "gliner" else None,
            "model_id": MODELS[name]["id"], "versions": VERSIONS,
            "prompt": None, "version_prompt": None}
        metadata_path = folder/"metadata.json"
        if not metadata_path.exists():
            atomic_json(metadata_path, metadata)
        else:
            prior = load_json(metadata_path)
            for k in ["revision_model", "revision_tokenizer", "labels", "effective_encoded_limit", "model_id", "versions"]:
                assert prior[k] == metadata[k], f"Model metadata differs: {k}"
        covered, totals = scan_model(name)
        started = time.perf_counter()
        failures = totals["error"]+totals["partial_invalid"]
        last = progress(name, len(covered), len(SAMPLE), failures, started, 0)
        try:
            for start in range(0, len(SAMPLE), CHECKPOINT_PARAGRAPHS):
                path = shard_path(name, start)
                if path.exists():
                    continue
                ACTIVE_RAW_FILE = path.relative_to(RUN_DIR).as_posix()
                records = []
                for local, p in enumerate(SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]):
                    ACTIVE_RAW_INDEX = local
                    result = process_paragraph(p, name, predict, measure, limit)
                    records.append(result)
                    totals[result["status_processing"]] += 1
                    covered.add(p["sample_position"])
                    failures += result["status_processing"] != "success"
                    last = progress(name, len(covered), len(SAMPLE), failures, started, last)
                atomic_json(path, {"schema_version": SCHEMA_VERSION, "model": name,
                    "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                    "payload_sha256": digest(encoded(records)), "results": records})
                del records
        finally:
            active_error = sys.exc_info()[1]
            attempt = {"model": name, "session_id": SESSION_ID, "load_seconds": metadata["load_seconds"],
                "inference_session_seconds": time.perf_counter()-started,
                "committed_paragraphs": len(scan_model(name)[0]), "at_processed": utc_now(),
                "status_processing": "interrupted_or_failed" if active_error else "success",
                "error": f"{type(active_error).__name__}: {active_error}" if active_error else None}
            atomic_json(folder/f"attempt_{SESSION_ID}_{secrets.token_hex(4)}.json", attempt)
        MODEL_OUTPUTS[name] = {"processing_seconds": sum(
            r["processing_seconds"] for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS)
            for r in load_shard(name,start)["results"])}

    def import_legacy(name):
        folder = RUN_DIR/"models"/name
        folder.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(LEGACY["path"]) as z:
            old = json.loads(z.read(LEGACY["members"][name]))
        assert old["source"] == SOURCE_META
        assert old["model_id"] == MODELS[name]["id"]
        assert old["metadata"]["revision_model"] == MODELS[name]["revision"]
        assert old["metadata"]["window"] == WINDOWS[name]
        if name == "gliner":
            assert old["metadata"]["threshold"] == GLINER_THRESHOLD
            assert old["metadata"]["labels"] == GLINER_LABELS
        assert len(old["results"]) == len(SAMPLE)
        if not (folder/"metadata.json").exists():
            atomic_json(folder/"metadata.json", {**old["metadata"], "model_id": old["model_id"],
                "imported_legacy": True, "legacy_sha256": LEGACY["sha256"]})
        for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS):
            if shard_path(name,start).exists():
                load_shard(name,start)
                continue
            results = old["results"][start:start+CHECKPOINT_PARAGRAPHS]
            for local,r in enumerate(results):
                assert r["paragraph"] == SAMPLE[start+local]
                for m in r["mentions"]:
                    m["legacy_ref_output_raw"] = list(m["ref_output_raw"])
                    m["ref_output_raw"] = [
                        shard_path(name,start).relative_to(RUN_DIR).as_posix()+f"#/results/{local}/raw/"+ref.split("/raw/",1)[1]
                        for ref in m["ref_output_raw"]]
            atomic_json(shard_path(name,start), {"schema_version": SCHEMA_VERSION, "model": name,
                "sample_sha256": SAMPLE_HASH, "inference_sha256": INFERENCE_HASH,
                "payload_sha256": digest(encoded(results)), "results": results})
        del old

    if LEGACY:
        for name in MODELS:
            if not model_complete(name):
                import_legacy(name)
    for name in MODELS:
        print(name, "committed:", len(scan_model(name)[0]), "/", len(SAMPLE))


### 6 — בחירה

PER/DATE עוברים סף בחירה; TIME/EVENT עוברים לתוצר מועמדים נפרד.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("6 — selection and table utilities"):
    COLUMNS = ["entity_group", "score", "word", "start", "end", "Source", "Index", "Text", "subtype"]
    ARROW_SCHEMA = pa.schema([(k, pa.int64() if k in {"start", "end", "Index"}
        else pa.float64() if k == "score" else pa.string()) for k in COLUMNS])
    def table(rows):
        return pa.Table.from_pylist([{k: r.get(k) for k in COLUMNS} for r in rows], schema=ARROW_SCHEMA)
    def roundtrip(cp, pp):
        arrow = pq.read_table(pp)
        assert arrow.schema == ARROW_SCHEMA
        frame = pd.read_csv(cp, encoding="utf-8-sig", keep_default_na=False, dtype="string")
        assert list(frame.columns) == COLUMNS and len(frame) == arrow.num_rows
        for c, p in zip(frame.to_dict("records"), arrow.to_pylist()):
            for k in COLUMNS:
                if k in {"start", "end", "Index"}:
                    assert int(c[k]) == p[k]
                elif k == "score":
                    assert (c[k] == "" and p[k] is None) or (c[k] != "" and float(c[k]) == p[k])
                else:
                    assert c[k] == ("" if p[k] is None else p[k])
        return arrow.to_pylist()
    def write_tables(rows, cp, pp):
        arrow = table(rows)
        with Path(cp).open("x", encoding="utf-8-sig", newline="") as f:
            arrow.to_pandas().to_csv(f, index=False, columns=COLUMNS)
        with Path(pp).open("xb") as f:
            pq.write_table(arrow, f)
        assert roundtrip(cp, pp) == arrow.to_pylist()
    def span_key(m):
        return (m["Source"], m["Index"], m["entity_group"], m["start"], m["end"])
    def overlaps(a, b):
        return (a["Source"], a["Index"]) == (b["Source"], b["Index"]) and max(a["start"], b["start"]) < min(a["end"], b["end"])
    def select_mentions(mentions):
        valid = [m for m in mentions if m["status_validation"] == "valid"]
        support = defaultdict(list)
        for m in valid:
            support[span_key(m)].append(m)
        primary = [m for m in valid if m["entity_group"] in SELECTION_THRESHOLDS
            and PRIMARY[m["entity_group"]] == m["model"] and m["score"] is not None
            and m["score"] >= SELECTION_THRESHOLDS[m["entity_group"]]]
        gliner_per = [m for m in primary if m["entity_group"] == "PER"]
        additions = [m for m in valid if ADD_BERT_PER and m["model"] == "bert"
            and m["entity_group"] == "PER" and m["score"] is not None
            and m["score"] >= BERT_PER_MIN_SCORE and not any(overlaps(m, g) for g in gliner_per)]
        accepted = list(primary)
        filtered_additions = []
        for m in sorted(additions, key=lambda x: (-x["score"], x["start"], x["end"])):
            if not any(overlaps(m, x) for x in accepted if x["entity_group"] == "PER"):
                filtered_additions.append(m)
                accepted.append(m)
        additions = filtered_additions
        selected, seen = [], set()
        for m in primary + additions:
            key = span_key(m)
            if key in seen:
                continue
            seen.add(key)
            candidate = m["entity_group"] in {"TIME", "EVENT"}
            high = m["entity_group"] == "EVENT" and m["score"] is not None and m["score"] >= EVENT_HIGH_PRIORITY_SCORE
            selected.append({**m, "selected_model": m["model"], "selected_model_id": m["model_id"],
                "selection_score": m["score"], "selection_policy": "v2",
                "selection_reason": "bert_per_supplement" if m in additions else "primary_v2",
                "selection_status": "candidate_high_priority" if high else "candidate" if candidate else "selected_mention",
                "semantic_validation": "not_assessed", "verified_event_fact": False,
                "supporting_models": sorted({x["model"] for x in support[key]}),
                "support": [{k: x[k] for k in ["model", "model_id", "label_original", "subtype", "score", "ref_output_raw"]}
                            for x in support[key]]})
        return sorted(selected, key=lambda m: (m["sample_position"], m["start"], m["end"], m["entity_group"]))
    print("v2 ready: MISC=BERT; TIME/EVENT candidates; score is the chosen model's score.")

def select_candidates(mentions):
    answer, seen = [], set()
    for m in mentions:
        if m["model"] != "gliner" or m["entity_group"] not in {"TIME", "EVENT"} or m["status_validation"] != "valid":
            continue
        if span_key(m) in seen:
            continue
        seen.add(span_key(m))
        high = m["entity_group"] == "EVENT" and m["score"] is not None and m["score"] >= EVENT_HIGH_PRIORITY_SCORE
        answer.append({**m, "selection_status": "candidate_high_priority" if high else "candidate",
            "selected_model": "gliner", "semantic_validation": "not_assessed", "verified_event_fact": False})
    return answer


### 7 — בדיקות

בדיקות חלונות, ספים, מועמדים, היסטים ושמירה.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("7 — v2 selftests"):
    original_data, original_meta = DATA, SOURCE_META
    saved_thresholds, saved_add = SELECTION_THRESHOLDS, ADD_BERT_PER
    SELECTION_THRESHOLDS = {"PER":.5,"DATE":.5,"LOC":0.,"ORG":0.,"MISC":0.}
    ADD_BERT_PER = True
    try:
        t = "Paris Paris John Smith battle 1941"
        DATA = {"test": "prefix:"+t}
        SOURCE_META = {"sha256":"test", "commit_sha":"test"}
        p = {"Source":"test", "Index":0, "Text":t, "start_document":7,
            "end_document":7+len(t), "sample_position":0}
        def m(name,label,a,b,score):
            return mention(p,name,{"label":label,"start":a,"end":b,"text":t[a:b],"score":score},0,"test")
        items=[m("bert","LOC",0,5,.8),m("bert","LOC",6,11,.8),
            m("gliner","person",12,22,.7),m("bert","PER",12,16,.99),
            m("gliner","person",0,5,.1),m("gliner","date",30,34,.1),
            m("gliner","event",23,29,.95)]
        chosen=select_mentions(items)
        assert len([x for x in chosen if x["entity_group"]=="LOC"])==2
        assert all(x["entity_group"] not in {"TIME","EVENT"} for x in chosen)
        assert not any(x["entity_group"]=="DATE" for x in chosen)
        assert not any(x["word"]=="John" for x in chosen)
        candidates=select_candidates(items)
        assert len(candidates)==1 and not candidates[0]["verified_event_fact"]
        bad=mention(p,"gliner",{"label":"person","text":t+".","start":0,"end":len(t)+1,"score":.99},0,"test")
        assert bad["status_validation"]=="invalid"
        text="a b c New York d e f g"
        ws=list(windows(text,{"max_words":4,"overlap_words":2,"max_encoded_tokens":999},len,999))
        assert any("New York" in w["Text"] for w in ws)
        with tempfile.TemporaryDirectory(dir=RUN_DIR) as tmp:
            write_tables(chosen,Path(tmp)/"test.csv",Path(tmp)/"test.parquet")
            write_tables([],Path(tmp)/"empty.csv",Path(tmp)/"empty.parquet")
        checks={"threshold_separation":True,"repeated_mentions":True,"per_overlap":True,
            "event_separate":True,"synthetic_punctuation_invalid":True,"boundary_window":True,
            "csv_parquet":True}
    finally:
        DATA,SOURCE_META=original_data,original_meta
        SELECTION_THRESHOLDS, ADD_BERT_PER = saved_thresholds, saved_add
    save_json(PATHS["selftests"],checks)
    print(checks)


### 8 — BERT

טעינה רק אם נותרו מקטעים; כל מקטע נשמר מיד.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("8 — bert"):
    if model_complete("bert"):
        print("bert: checkpoint coverage complete; model loading skipped")
    else:
        started_load = time.perf_counter()
        name = 'bert'
        revision = hf_revision(name)
        tokenizer = AutoTokenizer.from_pretrained(MODELS[name]["id"], revision=revision, use_fast=True)
        assert tokenizer.is_fast, "A fast tokenizer is required"
        model = AutoModelForTokenClassification.from_pretrained(MODELS[name]["id"], revision=revision)
        ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="average",
                       device=0 if DEVICE.startswith("cuda") else -1)
        limit = min(int(tokenizer.model_max_length), int(model.config.max_position_embeddings))
        print(name, "device:", ner.device, "revision:", revision, "token limit:", limit)
        load_seconds = time.perf_counter()-started_load
        print("BERT load seconds:", load_seconds)
        run_model(name, ner, tokenizer_measure(tokenizer), limit,
                  {"load_seconds": load_seconds, "revision_model": revision, "revision_tokenizer": revision,
                    "labels": sorted(set(model.config.id2label.values())),
                    "unsupported": ["DATE", "TIME", "EVENT"], "aggregation_strategy": "average"})
        print("BERT inference seconds:", MODEL_OUTPUTS[name]["processing_seconds"])
        del ner, model, tokenizer
        release()


### 9 — GLiNER

טעינה לאחר שחרור BERT; אין שמירת כל הפלט הגולמי בזיכרון.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("9 — gliner"):
    if model_complete("gliner"):
        print("gliner: checkpoint coverage complete; model loading skipped")
    else:
        started_load = time.perf_counter()
        revision = hf_revision("gliner")
        directory = snapshot_download(MODELS["gliner"]["id"], revision=revision)
        model = AutoExtractor.from_pretrained(directory).to(DEVICE)
        model.eval()
        model.strict_extraction = True  # שגיאות ה־API חייבות להישמר ככשל ולא כפלט ריק
        schema = model.create_schema().entities(GLINER_LABELS)
        limit = int(model.encoder.config.max_position_embeddings)
        def measure_gliner(text):
            # API שנבדק ב־2.1.0: max_len=None מונע חיתוך מילים
            return len(model.processor.transform_record(text, schema, max_len=None).input_ids)
        def predict_gliner(text):
            return model.extract_entities(text, GLINER_LABELS, threshold=GLINER_THRESHOLD,
                include_spans=True, include_confidence=True, max_len=None, overlap_policy="allow")
        assert "event" in GLINER_LABELS
        print("GLiNER device:", next(model.parameters()).device, "revision:", revision, "limit:", limit)
        load_seconds = time.perf_counter()-started_load
        print("GLiNER load seconds:", load_seconds)
        run_model("gliner", predict_gliner, measure_gliner, limit,
            {"load_seconds": load_seconds, "revision_model": revision, "revision_tokenizer": revision,
             "model_files_sha256": {str(p.relative_to(directory)): digest_file(p)
                 for p in sorted(Path(directory).rglob("*")) if p.is_file()}, "labels": GLINER_LABELS,
             "overlap_policy": "allow", "max_len": None, "unsupported": []})
        print("GLiNER inference seconds:", MODEL_OUTPUTS["gliner"]["processing_seconds"])
        del predict_gliner, measure_gliner, schema, model
        release()


### 10 — בחירה ויצוא

כתיבה זורמת ל־JSON/CSV/Parquet ותוצר מועמדים נפרד.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("10 — streaming selection and export"):
    assert all(model_complete(n) for n in MODELS), "Incomplete inference coverage"
    CANDIDATE_COLUMNS = COLUMNS+["selection_status", "selected_model", "semantic_validation", "verified_event_fact"]
    CANDIDATE_SCHEMA = pa.schema(list(ARROW_SCHEMA)+[
        pa.field("selection_status",pa.string()), pa.field("selected_model",pa.string()),
        pa.field("semantic_validation",pa.string()),pa.field("verified_event_fact",pa.bool_())])
    OUTPUT_FILES = {}
    for kind in ["selected", "candidates"]:
        basename = RUN_NAME if kind == "selected" else f"{RUN_NAME}_candidates"
        for ext in ["csv", "parquet", "json"]:
            OUTPUT_FILES[kind+"_"+ext] = EXPORT_DIR/f"{basename}.{ext}"
        (EXPORT_DIR/(kind+"_parts")).mkdir()
    indexes = {k:[] for k in ["selected","candidates"]}
    EXPORT_COUNTS = Counter()
    EXPORT_LABELS = {k:Counter() for k in indexes}
    started,last = time.perf_counter(),0
    with contextmanager_export_writers() as writers:
        for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS):
            chunks={n:load_shard(n,start) for n in MODELS}
            rows={k:[] for k in indexes}
            for local,p in enumerate(SAMPLE[start:start+CHECKPOINT_PARAGRAPHS]):
                mentions=[m for n in MODELS for m in chunks[n]["results"][local]["mentions"]]
                selected=select_mentions(mentions)
                candidates=select_candidates(mentions)
                for m in selected+candidates:
                    assert m["Text"] == p["Text"]
                    assert m["Text"][m["start"]:m["end"]] == m["word"]
                    assert m["start_document"] == p["start_document"]+m["start"]
                    assert m["end_document"] == p["start_document"]+m["end"]
                    assert DATA[m["Source"]][m["start_document"]:m["end_document"]] == m["word"]
                rows["selected"].extend(selected);rows["candidates"].extend(candidates)
                EXPORT_COUNTS["bert_per_additions"]+=sum(m["selection_reason"]=="bert_per_supplement" for m in selected)
            for kind,values in rows.items():
                cols, schema, csv_writer, parquet_writer = writers[kind]
                csv_writer.writerows({k:m.get(k) for k in cols} for m in values)
                parquet_writer.write_table(pa.Table.from_pylist([{k:m.get(k) for k in cols} for m in values],schema=schema))
                part = EXPORT_DIR/(kind+"_parts")/f"part_{start:08d}.json"
                atomic_json(part,{"schema_version":SCHEMA_VERSION,"sample_sha256":SAMPLE_HASH,
                    "kind":kind,"records":values})
                indexes[kind].append({"path":part.relative_to(EXPORT_DIR).as_posix(),
                    "record_count":len(values),"sha256":digest_file(part)})
                EXPORT_COUNTS[kind]+=len(values)
                EXPORT_LABELS[kind].update(m["entity_group"] for m in values)
            last=progress("selection/export",min(start+CHECKPOINT_PARAGRAPHS,len(SAMPLE)),len(SAMPLE),0,started,last)
    for kind in indexes:
        atomic_json(OUTPUT_FILES[kind+"_json"],{"schema_version":SCHEMA_VERSION,
            "run_name":RUN_NAME,"sample_sha256":SAMPLE_HASH,"source":SOURCE_META,
            "configuration_sha256":CONFIG_HASH,"selection_configuration":CONFIG,
            "record_count":EXPORT_COUNTS[kind],"parts":indexes[kind],
            "semantic_validation":"not_assessed"})
    print("Export counts:",dict(EXPORT_COUNTS))


### 11 — אימות

בדיקת כיסוי מלא, היסטים, round-trip וסטטוס שאינו מסתיר כשלים.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("11 — coverage, offsets and persisted outputs"):
    summary={}
    for name in MODELS:
        covered,totals=scan_model(name)
        assert covered==set(range(len(SAMPLE)))
        assert sum(totals[s] for s in ["success","partial_invalid","error"])==len(SAMPLE)
        counts=Counter();labels=Counter();started,last,failures=time.perf_counter(),0,0
        for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS):
            chunk=load_shard(name,start)
            raw_file=shard_path(name,start).relative_to(RUN_DIR).as_posix()
            for local,r in enumerate(chunk["results"]):
                p=r["paragraph"]
                failures+=r["status_processing"]!="success"
                counts["duplicates_resolved"]+=r.get("duplicates_resolved",0)
                counts["error_messages"]+=len(r["errors"])
                for m in r["mentions"]:
                    counts[m["status_validation"]]+=1
                    labels[m["entity_group"] or "UNMAPPED"]+=1
                    if m["status_validation"]=="valid":
                        assert p["Text"][m["start"]:m["end"]]==m["word"]
                        assert m["start_document"]==p["start_document"]+m["start"]
                        assert m["end_document"]==p["start_document"]+m["end"]
                        assert DATA[p["Source"]][m["start_document"]:m["end_document"]]==m["word"]
                    else:
                        assert m["error"]
                    for ref in m["ref_output_raw"]:
                        filename,pointer=ref.split("#",1)
                        assert filename==raw_file
                        node=chunk
                        for token in pointer.strip("/").split("/"):
                            node=node[int(token)] if isinstance(node,list) else node[token]
                        assert node is not None
                last=progress("audit_"+name,p["sample_position"]+1,len(SAMPLE),failures,started,last)
        for key in ["valid","pending","invalid","error_messages","duplicates_resolved"]:
            counts.setdefault(key,0)
        inference_seconds=sum(r["processing_seconds"] for start in range(0,len(SAMPLE),CHECKPOINT_PARAGRAPHS)
            for r in load_shard(name,start)["results"])
        meta=load_json(RUN_DIR/"models"/name/"metadata.json")
        summary[name]={"planned":len(SAMPLE),"covered":len(covered),"coverage":len(covered)/len(SAMPLE),
            "success":totals["success"],"partial_invalid":totals["partial_invalid"],"error":totals["error"],
            "mentions_by_label":dict(labels),"counts":dict(counts),"inference_seconds":inference_seconds,
            "initial_load_seconds":meta.get("load_seconds"),"metadata":meta}
        summary[name]["attempts"]=[load_json(p) for p in sorted((RUN_DIR/"models"/name).glob("attempt_*.json"))]
    for kind in ["selected","candidates"]:
        index=load_json(OUTPUT_FILES[kind+"_json"])
        assert index["sample_sha256"]==SAMPLE_HASH and index["source"]==SOURCE_META
        actual=0
        for part in index["parts"]:
            path=EXPORT_DIR/part["path"]
            assert digest_file(path)==part["sha256"]
            saved=load_json(path)
            assert saved["sample_sha256"]==SAMPLE_HASH
            assert len(saved["records"])==part["record_count"]
            keys=set()
            for m in saved["records"]:
                assert span_key(m) not in keys
                keys.add(span_key(m))
                assert m["status_validation"]=="valid"
                assert m["Text"][m["start"]:m["end"]]==m["word"]
                if kind=="selected":
                    assert m["entity_group"] not in {"TIME","EVENT"}
                    assert m["score"]==m["selection_score"]
                else:
                    assert m["entity_group"] in {"TIME","EVENT"}
                    assert m["selection_status"].startswith("candidate") and not m["verified_event_fact"]
            actual+=len(saved["records"])
        assert actual==index["record_count"]==EXPORT_COUNTS[kind]
        assert streaming_roundtrip(kind)==actual
    RUN_STATUS="completed_with_errors" if any(s["partial_invalid"] or s["error"] or s["counts"]["pending"] or s["counts"]["invalid"] for s in summary.values()) else "completed"
    atomic_json(EXPORT_DIR/"validation_summary.json",{"schema_version":SCHEMA_VERSION,
        "sample_sha256":SAMPLE_HASH,"run_status":RUN_STATUS,"mode":CONFIG["mode"],
        "eligible":TOTAL_ELIGIBLE,"processed":len(SAMPLE),"models":summary,
        "exports":dict(EXPORT_COUNTS),"selected_by_label":dict(EXPORT_LABELS["selected"]),
        "candidate_by_label":dict(EXPORT_LABELS["candidates"]),"selftests":load_json(PATHS["selftests"])})
    print("Run status:",RUN_STATUS,"models:",json.dumps(summary,indent=2),"exports:",dict(EXPORT_COUNTS))


### 12 — ZIP

manifest עם זמנים ו־hashes, אימות ארכיון והורדה.

פלט צפוי: SUCCESS וזמן; הודעת FAILED עם חריגה במקרה כשל.


In [ ]:
with cell("12 — ZIP preparation and verification"):
    assert all(model_complete(n) for n in MODELS)
    # Include source/checkpoints once, only the current export, and session selftests.
    artifacts=[RUN_DIR/"source.json",RUN_DIR/"sample.json",RUN_DIR/"run.json",PATHS["selftests"]]
    artifacts+=sorted(p for p in (RUN_DIR/"models").rglob("*.json") if p.is_file())
    artifacts+=sorted(p for p in EXPORT_DIR.rglob("*") if p.is_file())
    members={p.relative_to(RUN_DIR).as_posix():p for p in artifacts}
    assert len(members)==len(artifacts)
    with zipfile.ZipFile(ZIP_PATH,"x",zipfile.ZIP_DEFLATED) as z:
        for member,path in members.items():
            z.write(path,f"{RUN_NAME}/{member}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        assert z.testzip() is None
        for member,path in members.items():
            with z.open(f"{RUN_NAME}/{member}") as f:
                assert stream_digest(f)==digest_file(path)
try:
    manifest={"schema_version":SCHEMA_VERSION,"run_name":RUN_NAME,"run_status":RUN_STATUS,
        "mode":CONFIG["mode"],"configuration":CONFIG,"configuration_sha256":CONFIG_HASH,
        "inference_sha256":INFERENCE_HASH,"sample_sha256":SAMPLE_HASH,"source":SOURCE_META,
        "legacy":LEGACY,"models":summary,"versions":VERSIONS,"cell_timings":CELL_TIMINGS,
        "total_session_measured_seconds":sum(t["seconds"] for t in CELL_TIMINGS),
        "selection_export_seconds":sum(t["seconds"] for t in CELL_TIMINGS if t["title"].startswith("10 —")),
        "inference_seconds_all_checkpoints":{n:summary[n]["inference_seconds"] for n in MODELS},
        "timing_scope":"cell blocks including ZIP preparation; excludes manifest bookkeeping/final verification/browser transfer",
        "exports":dict(EXPORT_COUNTS),"artifact_sha256":{k:digest_file(p) for k,p in members.items()},
        "artifact_paths":{k:str(p) for k,p in members.items()},"zip_path":str(ZIP_PATH)}
    manifest_path=EXPORT_DIR/"manifest.json"
    atomic_json(manifest_path,manifest)
    member=manifest_path.relative_to(RUN_DIR).as_posix()
    with zipfile.ZipFile(ZIP_PATH,"a",zipfile.ZIP_DEFLATED) as z:
        z.write(manifest_path,f"{RUN_NAME}/{member}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        assert z.testzip() is None
        expected={f"{RUN_NAME}/{k}" for k in members}|{f"{RUN_NAME}/{member}"}
        assert set(z.namelist())==expected and len(z.namelist())==len(expected)
        assert z.read(f"{RUN_NAME}/{member}")==manifest_path.read_bytes()
        for key,want in manifest["artifact_sha256"].items():
            with z.open(f"{RUN_NAME}/{key}") as f:
                assert stream_digest(f)==want
    print_cell_timings()
    print("Status:",RUN_STATUS,"ZIP:",ZIP_PATH,"SHA256:",digest_file(ZIP_PATH))
    files.download(str(ZIP_PATH))
    print("SUCCESS manifest and final ZIP verified; download requested")
except BaseException as exc:
    print(f"FAILED final manifest/archive: {type(exc).__name__}: {exc}")
    raise
